# Zonal crop data processing

This notebook contains the methodology for processing crop data contained in [paper]. The objective of this analysis to map crop production across the most common 172 crops onto the world's countries.

In [2]:
import os
from rasterstats import zonal_stats
from allfed_spatial.features.io import load_features, write_features

### Resources
The raw datasets used in this analysis are as follows:

In [49]:
EARTHSTAT_CROP_DATA = ""
# COUNTRY_BOUNDARY_DATA = "https://osf.io/p93yg/download"
COUNTRY_BOUNDARY_DATA = "/Users/tim/allfed/residues_ruminants_project/gaul_all_countries_dissolved_4326.gpkg"

### Spatially join crop production with country boundaries using `zonal_stats`

`zonal_stats` from the `rasterstats` library gives us an easy way to summarise raster statistics across an iterable of vector geometries. We'll use this to create attributes on our country boundaries which describe the production of each of the 172 crops we have data for.

In [50]:
# Download and unzip crop data (note: requires ~3GB of storage)
#TODO download/unzip
base_crop_dir = '/Users/tim/allfed/data/landuse_yield/earthstat/HarvestedAreaYield175Crops_Geotiff/GeoTiff'
crop_dirs = [dirs for subdir, dirs, files in os.walk(base_crop_dir)][0]

In [55]:
# Download country boundaries and map to geometries
countries = load_features(COUNTRY_BOUNDARY_DATA)
country_geometries = [c.geom for c in countries]

In [56]:
# Iterate through crop folder, and run the production raster in each through
# zonal_stats on our country boundaries, assigning the result to the associated
# countries feature data.
for i, crop_type in enumerate(crop_dirs):
    print(f'Processing {i+1}/{len(crop_dirs)}: {crop_type}')
    stats = zonal_stats(
        country_geometries,
        '{}/{}/{}_Production.tif'.format(base_crop_dir, crop_type, crop_type),
        stats=['sum']
    )
    for j, s in enumerate(stats):
        countries[j].update_data(
            f'{crop_type}_sum', 
            round(s['sum'] if s['sum'] else 0.0, 3)
        )

Processing 1/172: apple


In [45]:
# Write data
write_features(countries, "../data/processed/global_crop_boundaries.gpkg")